In [ ]:
import numpy as np
import pandas as pd
import sys
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit
import sys
sys.path.insert(0, '../../')
from openabc.forcefields.parsers import HPSParser
from openabc.forcefields import CALVADOSModel
from openabc.utils.helper_functions import build_straight_CA_chain, write_pdb
from openabc.lib import _kcal_to_kj, kB, NA, VEP, EC

In [ ]:
# build pdb file
seq = 'VPGVG' * 2
ca_atoms = build_straight_CA_chain(seq, r0=0.38)
ca_pdb = 'ca_chain.pdb'
write_pdb(ca_atoms, ca_pdb)

# parse pdb file and update some parameters based on CALVADOS settings
protein_parser = HPSParser(ca_pdb)
protein_parser.protein_bonds.loc[:, 'k_bond'] = 8033.0
flag = protein_parser.atoms['resname'] == 'HIS'
protein_parser.atoms.loc[flag, 'charge'] = 0.0
assert len(protein_parser.atoms['chainID'].unique()) == 1 # ensure only one chain
charge = protein_parser.atoms['charge'].to_numpy()
charge[0] += 1
charge[-1] -= 1
protein_parser.atoms['charge'] = charge
T = 300 * unit.kelvin
T_K = T.value_in_unit(unit.kelvin)
dielectric = 5321 / T_K + 233.76 - 0.9297 * T_K + 1.417e-3 * T_K**2 - 8.292e-7 * T_K**3
print(f'Set temperature-dependent dielectric as {dielectric:.6f}')

In [ ]:
# set up CALVADOS model
model = CALVADOSModel()
model.append_mol(protein_parser)
top = app.PDBFile(ca_pdb).getTopology()
model.create_system(top=top, box_a=1000, box_b=1000, box_c=1000)
model.add_protein_bonds(force_group=1)
model.add_contacts('CALVADOS2', cutoff=2.0*unit.nanometer, force_group=2)
ionic_strength = 150 * unit.millimolar
ldby = (kB * T * VEP * dielectric / (2 * NA * ionic_strength * EC**2))**0.5
model.add_dh_elec(ldby, dielectric, cutoff=4.0*unit.nanometer, force_group=3)

In [ ]:
# set up and run simulation
# run a short trajectory on CPU for test
# in production run, use GPU for better performance
friction_coeff = 1.0 / unit.picosecond
timestep = 10.0 * unit.femtosecond
integrator = mm.LangevinMiddleIntegrator(T, friction_coeff, timestep)
init_coord = app.PDBFile(ca_pdb).getPositions()
model.set_simulation(integrator, platform_name='CPU', init_coord=init_coord)
model.simulation.minimizeEnergy()
model.simulation.context.setVelocitiesToTemperature(T)
output_dcd = 'output.dcd'
model.add_reporters(report_interval=100, output_dcd=output_dcd)
model.simulation.step(500)